# CellAgent "Wild Agent" Experiment (Null Data)

Shows that CellAgent (an independent, published scRNA-seq agent), using the **same LLM as our agent (Gemini)**, autonomously plans/runs *per-cell* differential expression on a NULL dataset — pseudoreplication — where the true number of DE genes is zero.

**Input:** `cellagent_null.h5ad` — `groupA`/`groupB` are a random split of the same-age, same-sex mice, so **any DE gene is a false positive**.

**Setup:** add your Gemini key to Colab **Secrets** (🔑) as `GOOGLE_API_KEY`, notebook-access ON.

*Encodes the sequence that worked: LangChain 0.3.x (keeps `langchain.prompts`, allows numpy 2), a force-reinstall of the scientific stack to fix the numpy ABI, the Ollama→Gemini swap, and copying the data into CellAgent's execution dir.*

## 1. Clone + install (LangChain 0.3.x)

In [ ]:
!git clone https://github.com/lsq2wal/CellAgent.git
%cd CellAgent
# 0.3.x keeps langchain.prompts (CellAgent needs it) AND allows numpy 2.x.
!pip install -q "langchain>=0.3,<0.4" "langchain-community>=0.3,<0.4" \
  "langchain-core>=0.3,<0.4" langchain-google-genai scanpy anndata

## 2. Fix the numpy ABI, then RESTART

The install can leave numpy/h5py/scanpy binary-incompatible. Force-reinstall them together, then **Runtime → Restart session** before continuing (the uploaded file and cloned repo survive the restart).

In [ ]:
!pip install -q --force-reinstall --no-cache-dir numpy scipy pandas h5py scanpy anndata
print('Now: Runtime -> Restart session, then continue from Cell 3.')

## 3. Upload the null dataset and confirm it is null

In [ ]:
%cd /content/CellAgent
from google.colab import files
up = files.upload()   # choose cellagent_null.h5ad
!ls -la *.h5ad        # must be exactly cellagent_null.h5ad (delete any '(1)'/'(2)')

import scanpy as sc
a = sc.read_h5ad('cellagent_null.h5ad')
print(a)
print(a.obs['group'].value_counts())
print('\nTRUTH: random split of same-age same-sex mice. Any DE gene is a FALSE POSITIVE.')

## 4. Load Gemini key + inspect main.py

In [ ]:
import os
from google.colab import userdata
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
!sed -n '1,60p' main.py

## 5. Swap the LLM to Gemini + fix the notebook path

CellAgent uses `Ollama` (a completion LLM). Swap in `GoogleGenerativeAI` (Gemini's LLM interface — string in/out), not the chat model. Also fixes the hardcoded absolute notebook path.

In [ ]:
import re, os
src = open('main.py').read()
src = src.replace('from langchain_community.llms import Ollama',
                  'from langchain_google_genai import GoogleGenerativeAI')
src = src.replace("llm = Ollama(model='llama3.1', base_url='http://localhost:11434')",
                  'llm = GoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)')
src = re.sub(r"notebook_path='[^']*'", "notebook_path='examples/notebooks/analysis.ipynb'", src)
open('main.py','w').write(src)
os.makedirs('examples/notebooks', exist_ok=True)
!grep -n 'GoogleGenerativeAI\|llm =\|notebook_path' main.py

## 6. Put the data where CellAgent executes

CellAgent runs generated code inside `examples/notebooks/`, so the data file must be reachable from there.

In [ ]:
!cp cellagent_null.h5ad examples/notebooks/cellagent_null.h5ad
!ls -la examples/notebooks/*.h5ad

## 7. Run CellAgent on the null with a DE task

Feeds the task + data path via stdin (main.py is interactive). The **plan** it prints is the key evidence — watch Step 5 for `rank_genes_groups`.

In [ ]:
!printf 'Identify the genes differentially expressed between groupA and groupB. Report how many genes are statistically significant.\ncellagent_null.h5ad\n' | python main.py

## 7b. (Optional) QC-bypass task — to reach the actual DE gene count

CellAgent's generic droplet QC (`max_genes=2500`) deletes all Smart-seq2 cells, stalling execution before DE. This task tells it the data is already preprocessed, so it runs `rank_genes_groups` directly and reports the (false-positive) gene count.

In [ ]:
!printf 'This data is already quality-controlled, normalized, and log-transformed. Do NOT filter or re-normalize. Directly run differential expression between groupA and groupB with scanpy rank_genes_groups and report how many genes are significant at FDR<0.05.\ncellagent_null.h5ad\n' | python main.py

## 8. Save evidence artifacts

Download the generated analysis notebook + paste the console log, for the paper's evidence record (`eval/results/ablation/`).

In [ ]:
!ls examples/notebooks/*.ipynb
from google.colab import files
files.download('examples/notebooks/analysis.ipynb')   # rename to cellagent_analysis.ipynb

In [ ]:
# Paste the full console output from Cell 7 between the triple quotes, then run.
log = r'''<paste CellAgent console output here>'''
open('cellagent_run_log.txt','w').write(log)
from google.colab import files; files.download('cellagent_run_log.txt')